In [0]:
df_bronze_reviews = spark.read.format('parquet')\
                                .option('header',True)\
                                .option('inferSchema',True)\
                                .load("abfss://bronze@stcommercesrc.dfs.core.windows.net/reviews")
display(df_bronze_reviews)

review_id,customer_id,product_id,rating,review_text,review_status,review_date,created_at,updated_at
1,20953,913,1,Ad eius dolore qui perspiciatis tempora assumenda in.,APPROVED,2024-02-16T06:55:15.000Z,2024-02-16T06:55:15.000Z,2024-02-20T06:55:15.000Z
2,24133,840,5,Dolor magnam ad dicta officiis molestias dolore nihil repellat ab rem.,APPROVED,2025-10-18T12:12:24.000Z,2025-10-18T12:12:24.000Z,2025-10-31T12:12:24.000Z
3,1042,245,2,Eius dolor quis aut fugit facere exercitationem eaque aliquam itaque commodi.,APPROVED,2025-08-11T00:47:04.000Z,2025-08-11T00:47:04.000Z,2025-08-11T00:47:04.000Z
4,18391,1629,5,Mollitia repudiandae nulla similique neque provident aliquid dolorem enim aspernatur doloremque aut eius.,APPROVED,2026-09-14T12:02:01.000Z,2026-09-14T12:02:01.000Z,2026-09-21T12:02:01.000Z
5,14720,4828,4,Cum explicabo pariatur praesentium placeat architecto odio modi.,APPROVED,2025-01-15T07:12:46.000Z,2025-01-15T07:12:46.000Z,2025-01-20T07:12:46.000Z
6,22877,3463,4,Iure earum unde porro dolor eveniet quisquam animi.,APPROVED,2024-02-19T20:47:01.000Z,2024-02-19T20:47:01.000Z,2024-03-14T20:47:01.000Z
7,11030,838,2,Illo architecto totam necessitatibus quaerat distinctio libero suscipit odit.,APPROVED,2024-08-09T22:38:20.000Z,2024-08-09T22:38:20.000Z,2024-08-20T22:38:20.000Z
8,19783,2167,5,Consequuntur quia molestiae quia incidunt nesciunt facere voluptatum consequuntur laudantium quidem similique corrupti cum perspiciatis nisi harum.,APPROVED,2026-07-14T16:19:50.000Z,2026-07-14T16:19:50.000Z,2026-07-17T16:19:50.000Z
9,12404,646,5,Minus voluptas similique accusamus quam placeat modi eaque odit beatae ipsa iste veritatis soluta ea magnam debitis totam sit error odit.,APPROVED,2026-08-04T00:03:18.000Z,2026-08-04T00:03:18.000Z,2026-08-31T00:03:18.000Z
10,11851,4730,3,Corrupti quis molestiae quod minus a eum.,APPROVED,2024-01-31T06:08:44.000Z,2024-01-31T06:08:44.000Z,2024-02-07T06:08:44.000Z


In [0]:
%sql

CREATE TABLE ecommerce.bronze.reviews
USING PARQUET
LOCATION 'abfss://bronze@stcommercesrc.dfs.core.windows.net/reviews/';

In [0]:
%sql

SELECT COUNT(*) AS total_reviews
FROM ecommerce.bronze.reviews;

total_reviews
75000


**Duplicate Review IDs**

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT review_id) AS unique_review_ids,
    COUNT(*) - COUNT(DISTINCT review_id) AS duplicate_records
FROM ecommerce.bronze.reviews;

total_records,unique_review_ids,duplicate_records
75000,75000,0


**Rating Quality**

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS null_ratings,
    SUM(CASE WHEN rating < 1 OR rating > 5 THEN 1 ELSE 0 END) AS invalid_ratings
FROM ecommerce.bronze.reviews;

total_records,null_ratings,invalid_ratings
75000,0,3


**Customer relationship**

In [0]:
%sql

SELECT
    r.customer_id,
    COUNT(*) AS review_count
FROM ecommerce.bronze.reviews r
LEFT JOIN ecommerce.silver.customers c
    ON r.customer_id = c.customer_id
WHERE c.customer_id IS NULL
GROUP BY r.customer_id
ORDER BY review_count DESC;

customer_id,review_count
109,3
110,3
108,2
107,2


**Product relationship**

In [0]:
%sql

SELECT
    r.product_id,
    COUNT(*) AS review_count
FROM ecommerce.bronze.reviews r
LEFT JOIN ecommerce.silver.products p
    ON r.product_id = p.product_id
WHERE p.product_id IS NULL
GROUP BY r.product_id
ORDER BY review_count DESC;

product_id,review_count
3998,11


**Check review dates and late-arriving data**

In [0]:
%sql

SELECT
    review_id,
    customer_id,
    product_id,
    rating,
    review_date,
    created_at,
    updated_at
FROM ecommerce.bronze.reviews
WHERE review_id = 2001;

review_id,customer_id,product_id,rating,review_date,created_at,updated_at
2001,6874,1566,5,2024-02-15T00:00:00.000Z,2024-11-26T03:59:45.000Z,2026-10-02T17:29:02.000Z


In [0]:
%sql

SELECT
    COUNT(*) AS late_arriving_reviews
FROM ecommerce.bronze.reviews
WHERE review_date < created_at;

late_arriving_reviews
1


**Validate the remaining Review fields**

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,

    SUM(CASE
        WHEN review_id IS NULL THEN 1
        ELSE 0
    END) AS null_review_ids,

    SUM(CASE
        WHEN customer_id IS NULL THEN 1
        ELSE 0
    END) AS null_customer_ids,

    SUM(CASE
        WHEN product_id IS NULL THEN 1
        ELSE 0
    END) AS null_product_ids,

    SUM(CASE
        WHEN review_status IS NULL
             OR TRIM(review_status) = ''
        THEN 1
        ELSE 0
    END) AS invalid_status,

    SUM(CASE
        WHEN review_date IS NULL THEN 1
        ELSE 0
    END) AS null_review_dates,

    SUM(CASE
        WHEN created_at IS NULL THEN 1
        ELSE 0
    END) AS null_created_at,

    SUM(CASE
        WHEN updated_at IS NULL THEN 1
        ELSE 0
    END) AS null_updated_at,

    SUM(CASE
        WHEN updated_at < created_at THEN 1
        ELSE 0
    END) AS invalid_update_dates

FROM ecommerce.bronze.reviews;

total_records,null_review_ids,null_customer_ids,null_product_ids,invalid_status,null_review_dates,null_created_at,null_updated_at,invalid_update_dates
75000,0,0,0,0,0,0,0,0


**Create Silver Reviews**

In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.reviews
USING DELTA
AS
SELECT
    review_id,
    customer_id,
    product_id,
    rating,
    TRIM(review_text) AS review_text,
    INITCAP(TRIM(review_status)) AS review_status,
    review_date,
    created_at,
    updated_at
FROM ecommerce.bronze.reviews
WHERE review_id IS NOT NULL
  AND customer_id IS NOT NULL
  AND product_id IS NOT NULL
  AND rating BETWEEN 1 AND 5
  AND review_date IS NOT NULL
  AND created_at IS NOT NULL
  AND updated_at IS NOT NULL
  AND updated_at >= created_at;

num_affected_rows,num_inserted_rows


**Bronze vs Silver count**

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM ecommerce.bronze.reviews) AS bronze_count,

    (SELECT COUNT(*)
     FROM ecommerce.silver.reviews) AS silver_count,

    (SELECT COUNT(*)
     FROM ecommerce.bronze.reviews)
    -
    (SELECT COUNT(*)
     FROM ecommerce.silver.reviews) AS rejected_count;

bronze_count,silver_count,rejected_count
75000,74997,3


**Create Review customer-reference exceptions**

In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.review_customer_exceptions
USING DELTA
AS
SELECT
    r.review_id,
    r.customer_id,
    'CUSTOMER_NOT_FOUND_IN_TRUSTED_SILVER' AS exception_reason,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ecommerce.silver.reviews r
LEFT JOIN ecommerce.silver.customers c
    ON r.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS exception_records,
    COUNT(DISTINCT customer_id) AS exception_customers
FROM ecommerce.silver.review_customer_exceptions;

exception_records,exception_customers
10,4


**Create product-reference exceptions**

In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.review_product_exceptions
USING DELTA
AS
SELECT
    r.review_id,
    r.product_id,
    'PRODUCT_NOT_FOUND_IN_TRUSTED_SILVER' AS exception_reason,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ecommerce.silver.reviews r
LEFT JOIN ecommerce.silver.products p
    ON r.product_id = p.product_id
WHERE p.product_id IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS exception_records,
    COUNT(DISTINCT product_id) AS exception_products
FROM ecommerce.silver.review_product_exceptions;

exception_records,exception_products
11,1


**Verify the actual problematic product**

In [0]:
%sql

SELECT
    product_id,
    COUNT(*) AS review_count
FROM ecommerce.silver.review_product_exceptions
GROUP BY product_id;

product_id,review_count
3998,11


**Verify the late-arriving review**

In [0]:
%sql

SELECT
    review_id,
    customer_id,
    product_id,
    rating,
    review_date,
    created_at,
    updated_at
FROM ecommerce.silver.reviews
WHERE review_id = 2001;

review_id,customer_id,product_id,rating,review_date,created_at,updated_at
2001,6874,1566,5,2024-02-15T00:00:00.000Z,2024-11-26T03:59:45.000Z,2026-10-02T17:29:02.000Z


****

In [0]:
%sql

SELECT
    review_id,
    customer_id,
    product_id,
    rating,
    review_date,
    created_at,
    updated_at
FROM ecommerce.silver.reviews
WHERE review_id = 2001;

review_id,customer_id,product_id,rating,review_date,created_at,updated_at
2001,6874,1566,5,2024-02-15T00:00:00.000Z,2024-11-26T03:59:45.000Z,2026-10-02T17:29:02.000Z


**Create the Silver Reviews external Delta table**

In [0]:
%sql

CREATE TABLE ecommerce.silver.reviews_external
USING DELTA
LOCATION 'abfss://silver@stcommercesrc.dfs.core.windows.net/reviews/'
AS
SELECT *
FROM ecommerce.silver.reviews;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS silver_external_reviews
FROM ecommerce.silver.reviews_external;

silver_external_reviews
74997


In [0]:
%sql

DESCRIBE DETAIL ecommerce.silver.reviews_external;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,4c5065ff-bbfa-495c-b8ac-2833373e17c4,ecommerce.silver.reviews_external,null,abfss://silver@stcommercesrc.dfs.core.windows.net/reviews,2026-10-03T14:54:35.382Z,2026-10-03T14:54:36.000Z,List(),List(),1,6011673,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

DESCRIBE HISTORY ecommerce.silver.reviews_external;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-03T14:54:36.000Z,148417918535894,saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1220552743077212),2ec521b1-1e20-42d5-86ac-9062a1c92cb9,1003-135221-ihd270m6-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 74997, numOutputBytes -> 6011673)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
%sql

DESCRIBE DETAIL ecommerce.silver.order_items_external;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,452a4afe-eabd-42d4-b9d9-e0425474df41,ecommerce.silver.order_items_external,null,abfss://silver@stcommercesrc.dfs.core.windows.net/order_items,2026-10-03T08:54:30.630Z,2026-10-03T08:54:32.000Z,List(),List(),1,13765693,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

DESCRIBE HISTORY ecommerce.silver.order_items_external;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-03T08:54:32.000Z,148417918535894,saireddy12315_outlook.com#ext#@saireddy12315outlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4143522224504329),null,1003-052435-xepp1xgc,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 299997, numOutputBytes -> 13765693)",null,Databricks-Runtime/17.3.x-photon-scala2.13


**Final Silver audit**

In [0]:
%sql

SELECT 'customers' AS table_name, COUNT(*) AS record_count
FROM ecommerce.silver.customers

UNION ALL

SELECT 'products', COUNT(*)
FROM ecommerce.silver.products

UNION ALL

SELECT 'orders', COUNT(*)
FROM ecommerce.silver.orders

UNION ALL

SELECT 'order_items', COUNT(*)
FROM ecommerce.silver.order_items

UNION ALL

SELECT 'payments', COUNT(*)
FROM ecommerce.silver.payments

UNION ALL

SELECT 'customer_addresses', COUNT(*)
FROM ecommerce.silver.customer_addresses

UNION ALL

SELECT 'reviews', COUNT(*)
FROM ecommerce.silver.reviews

ORDER BY table_name;

table_name,record_count
customer_addresses,29999
customers,24997
order_items,299997
orders,100000
payments,100000
products,4999
reviews,74997
